### Essential SQL Concepts for Freshers

In [0]:
USE CATALOG thebricklearning;
USE SCHEMA aianalyticsengineer;

### Aliasing (AS)

- Makes column names readable in result sets.
- Essential for clarity, especially in joins and aggregations.

In [0]:
SELECT c.customer_name AS name, o.total_amount AS amount
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id;


### ORDER BY + LIMIT / TOP

- Fetches top-N records (e.g., top 5 customers by spend).
- Used in dashboards, pagination, and ranking.

In [0]:
SELECT * FROM orders
ORDER BY total_amount DESC
LIMIT 5;


### DISTINCT
Used for deduplication, dimension extraction, or quick exploration.

In [0]:
SELECT DISTINCT customer_city FROM customers;

### IN, NOT IN, BETWEEN

- Enables concise filtering logic.
- Common in BI dashboards and reporting tools.

In [0]:
SELECT * FROM orders 
WHERE customer_id IN (1, 2);

In [0]:
SELECT * FROM orders 
WHERE total_amount BETWEEN 100 AND 250;

### String Operations on Columns

In [0]:
SELECT 
  UPPER(customer_name) AS upper_name,
  LENGTH(customer_name) AS name_length
FROM customers;

### DATE Functions

In [0]:
SELECT 
  order_date,
  YEAR(order_date) AS order_year,
  MONTH(order_date) AS order_month
FROM orders;


### NULL Handling – COALESCE, IS NULL

In [0]:
SELECT 
  customer_name,
  COALESCE(customer_email, 'N/A') AS email_info
FROM customers;

In [0]:
SELECT * FROM orders
WHERE total_amount IS NOT NULL;

### Subqueries

In [0]:
SELECT * FROM customers
WHERE customer_id IN (
  SELECT customer_id FROM orders WHERE total_amount > 200
);


### UNION / UNION ALL

In [0]:
SELECT customer_name FROM customers
UNION
SELECT product_name FROM order_items;


###  Basic Window Functions

In [0]:
SELECT 
  customer_id,
  total_amount,
  ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_rank
FROM orders;


### Duplicate removal from a table

In data engineering, removing duplicate rows is a critical task during data cleansing, ETL processing, and data quality enforcement — especially before moving from Bronze ➝ Silver ➝ Gold layers in a medallion architecture (e.g., on Databricks).

Understanding Duplicate Removal from a Table
We’ll walk through:

- What is a duplicate?
- How to detect duplicates
- How to remove duplicates (with and without row IDs)
- Best practice for deduplication

### 1. What is a Duplicate?

A duplicate is a row that is identical in one or more selected columns. It can be:
- Entire row is repeated
- Or based on a business key (e.g., same customer_id, order_id, etc.)

- Detecting Duplicates
- Example: Identify rows in orders with duplicate order_ids

In [0]:
SELECT 
  order_id,
  COUNT(*) AS cnt
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;


### A. Remove Exact Duplicates (All Columns Match)

- Removes rows that are completely identical.
- Fast and useful for small tables or logs with exact repeats.

In [0]:
CREATE TABLE orders_deduped AS
SELECT DISTINCT * FROM orders;


### B. Deduplicate Based on Business Key (e.g., order_id) – Keep Latest

- Uses window function ROW_NUMBER() to rank rows per order_id.
- Keeps only the latest record.
- Very useful in CDC (Change Data Capture) scenarios.

In [0]:
CREATE OR REPLACE TABLE orders_deduped_2 AS
SELECT * FROM (
  SELECT *, 
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS rn
  FROM orders
) t
WHERE rn = 1;


### C. Delete Duplicates In-Place (if table supports DELETE)

- Keeps only the first occurrence of each duplicate order_id.

In [0]:
DELETE FROM orders
WHERE rowid NOT IN (
  SELECT MIN(rowid)
  FROM orders
  GROUP BY order_id
);


Practice	Why It Matters
- Define the deduplication logic clearly (by which columns?)	Avoid accidental data loss
- Use window functions for fine control	Choose which duplicate to retain (latest, earliest)
- Prefer non-destructive approach	Write deduped data to new table (e.g., Silver layer)
- Always log or count before deleting	Helps with observability and auditing

In [0]:
CREATE OR REPLACE TEMP VIEW sample_values AS
SELECT * FROM VALUES
  (1, 50),
  (2, 40),
  (3, 40),
  (4, 30),
  (5, 20),
  (6, 20)
AS sample(id, value);

In [0]:
SELECT 
    id,
    value,
    RANK() OVER (partition by id ORDER BY value DESC) AS rnk
FROM sample_values;

In [0]:
SELECT 
    id,
    value,
    RANK() OVER (ORDER BY value DESC)       AS rnk,
    DENSE_RANK() OVER (ORDER BY value DESC) AS dense_rnk,
    RANK() OVER (PARTITION BY value ORDER BY id) AS rptnk
FROM sample_values;


In [0]:
SELECT 
      id,
      value,
      RANK() OVER (PARTITION BY value ORDER BY id) AS rnk
  FROM sample_values

In [0]:
WITH ranked AS (
  SELECT 
      id,
      value,
      RANK() OVER (PARTITION BY value ORDER BY id) AS rnk
  FROM sample_values
)
SELECT *
FROM ranked
WHERE rnk = 1;
